In [4]:
import requests
import pandas as pd
import time
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- CONFIGURATION ---
SYMBOLS = ["BTCUSDT", "ETHUSDT"] 
INTERVAL = "1h"
START_DATE = "2021-03-15 00:00:00"
END_DATE = "2025-12-01 00:00:00"

# --- CRÉATION D'UNE SESSION ROBUSTE ---
# Garde la connexion SSL ouverte et réessaie automatiquement en cas de micro-coupure
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

def download_binance_spot(symbol, start_date, end_date):
    start_ts = int(pd.Timestamp(start_date, tz='UTC').timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_date, tz='UTC').timestamp() * 1000)
    
    print(f"\n📥 Début du téléchargement pour {symbol} Spot...")
    
    all_klines = []
    current_start = start_ts
    limit = 1000
    
    while current_start < end_ts:
        url = f"https://api.binance.com/api/v3/klines?symbol={symbol}&interval={INTERVAL}&startTime={current_start}&endTime={end_ts}&limit={limit}"
        
        try:
            # On utilise session.get au lieu de requests.get
            response = session.get(url, timeout=10)
            
            if response.status_code != 200:
                print(f"❌ Erreur API: {response.status_code}. Pause de 2s...")
                time.sleep(2)
                continue # On réessaie la même date
                
        except requests.exceptions.RequestException as e:
            # Si le SSL plante, on ne crash pas le script, on patiente et on relance
            print(f"⚠️ Micro-coupure détectée. Tentative de reconnexion dans 3 secondes...")
            time.sleep(3)
            continue
            
        data = response.json()
        if not data:
            break
            
        all_klines.extend(data)
        
        # Le prochain start_ts est le close time de la dernière bougie + 1 ms
        current_start = data[-1][6] + 1
        
        # Anti-spam
        time.sleep(0.5)
        
    # --- FORMATAGE ---
    if not all_klines:
        print("⚠️ Aucune donnée téléchargée.")
        return

    columns = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'qav', 'num_trades', 'tbbav', 'tbqav', 'ignore']
    df = pd.DataFrame(all_klines, columns=columns)
    
    df['date'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    cols_to_float = ['open', 'high', 'low', 'close', 'volume']
    df[cols_to_float] = df[cols_to_float].astype(float)
    
    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
    df = df.sort_values('date').drop_duplicates('date')
    
    # --- EXPORT ---
    output_file = f"{symbol.lower()}_spot_1h.parquet"
    df.to_parquet(output_file, engine="pyarrow")
    print(f"✅ Terminé ! {len(df)} lignes sauvegardées dans {output_file}")

# --- LANCEMENT ---
for sym in SYMBOLS:
    download_binance_spot(sym, START_DATE, END_DATE)


📥 Début du téléchargement pour BTCUSDT Spot...
✅ Terminé ! 41317 lignes sauvegardées dans btcusdt_spot_1h.parquet

📥 Début du téléchargement pour ETHUSDT Spot...
✅ Terminé ! 41317 lignes sauvegardées dans ethusdt_spot_1h.parquet
